In [2]:
# 01 패키지
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
from pathlib import Path

In [3]:
# 02 랜덤 고정값
np.random.seed(42)
random.seed(42)

In [4]:
# 03 경로 설정
DATA_DIR = Path("../data")
OUTPUT_PATH = DATA_DIR / "12_profile_metrics.csv"

In [5]:
# 04 clients 데이터 불러오기
clients = pd.read_csv(DATA_DIR / "01_clients.csv", encoding="utf-8-sig")

# 컬럼명 앞뒤 공백 제거
clients.columns = clients.columns.str.strip()

# 빈 문자열을 NaN으로 변환
clients = clients.replace(r"^\s*$", np.nan, regex=True)

# 전체가 빈 행인 경우 제거
clients = clients.dropna(how="all")

# client_id가 없는 행 제거
clients = clients.dropna(subset=["client_id"])

# 인덱스 재정렬
clients = clients.reset_index(drop=True)

# 데이터 확인
print(clients.shape)
clients.head()

(1, 6)


,client_id,client_name,client_industry,client_brand_category,client_plan_type,client_created_at
0,cli-0001,(주) 라이언스윔,헬스/웰니스,스포츠웨어,Scale,2026-05-01 14:30:02


In [6]:
# 05 profile_metrics 데이터 생성

# 분석 기간
start_date = pd.to_datetime("2026-05-14")
end_date = pd.to_datetime("2026-06-14")

date_list = pd.date_range(start=start_date, end=end_date, freq="D")

profile_metric_rows = []
profile_metric_id = 1

for _, client in clients.iterrows():
    client_id = client["client_id"]

    # 초기 팔로워 수
    # 현재 화면 예시가 622명 정도였으므로 500~650 사이에서 시작
    followers_count = np.random.randint(520, 620)

    for current_date in date_list:
        # 캠페인 시작 전후로 성과가 살짝 올라가는 구간 설정
        # 실제 캠페인 시작일이 2026-06-04 근처라면 이 구간에서 수치가 조금 더 높게 나옴
        is_campaign_period_1 = pd.to_datetime("2026-05-24") <= current_date <= pd.to_datetime("2026-05-30")
        is_campaign_period_2 = pd.to_datetime("2026-06-06") <= current_date <= pd.to_datetime("2026-06-12")

        if is_campaign_period_1:
            new_followers = np.random.randint(5, 35)
            unfollowers = np.random.randint(0, 10)
            reach_count = np.random.randint(4000, 16000)

        elif is_campaign_period_2:
            new_followers = np.random.randint(5, 35)
            unfollowers = np.random.randint(0, 10)
            reach_count = np.random.randint(4000, 16000)

        else:
            new_followers = np.random.randint(0, 10)
            unfollowers = np.random.randint(0, 5)
            reach_count = np.random.randint(500, 7000)

        # 누적 팔로워 수 갱신
        followers_count = followers_count + new_followers - unfollowers

        # 팔로워 수가 음수가 되지 않도록 보정
        followers_count = max(followers_count, 0)

        # 노출 수는 도달 수보다 크거나 같아야 함
        impressions_count = int(reach_count * np.random.uniform(1.2, 2.2))

        # 프로필 방문 수는 도달 수의 일부
        profile_visit_count = int(reach_count * np.random.uniform(0.008, 0.035))

        # 최소 1 이상으로 보정
        profile_visit_count = max(profile_visit_count, 1)

        # 웹사이트 클릭 수는 프로필 방문 수보다 작거나 같아야 함
        website_click_count = int(profile_visit_count * np.random.uniform(0.03, 0.15))
        website_click_count = min(website_click_count, profile_visit_count)

        # 웹사이트 클릭률
        # 소수형 비율로 저장: 0.0785 = 7.85%
        # 팀원들이랑 통일해서 소수형 안하기로 함 그래서 *100 붙임
        website_click_rate = round(website_click_count / profile_visit_count, 4)*100

        profile_metric_rows.append({
            # 프로필 성과 ID
            # "profile_metric_id": f"promet-{profile_metric_id:04d}",
            "profile_metric_id": f"{profile_metric_id:04d}",

            # 고객사 ID
            "client_id": client_id,

            # 프로필 집계 일자
            "profile_metric_at": current_date.strftime("%Y-%m-%d %H:%M:%S"),

            # 프로필 팔로워 수
            "profile_followers_count": followers_count,

            # 신규 팔로워 수
            "profile_new_followers_count": new_followers,

            # 언팔로워 수
            "profile_unfollowers_count": unfollowers,

            # 프로필 도달 수
            "profile_reach_count": reach_count,
            # 프로필 노출 수
            "profile_impressions_count": impressions_count,
            # 프로필 방문 수
            "profile_visit_count": profile_visit_count,
            # 웹사이트 클릭 수
            "profile_website_click_count": website_click_count,
            # 웹사이트 클릭률
            "profile_website_click_rate": website_click_rate
        })

        profile_metric_id += 1

profile_metrics = pd.DataFrame(profile_metric_rows)

# 확인
profile_metrics.head()

,profile_metric_id,client_id,profile_metric_at,profile_followers_count,profile_new_followers_count,profile_unfollowers_count,profile_reach_count,profile_impressions_count,profile_visit_count,profile_website_click_count,profile_website_click_rate
0,0001,cli-0001,2026-05-14 00:00:00,574,7,4,3592,4870,43,1,2.33
1,0002,cli-0001,2026-05-15 00:00:00,577,7,4,3671,4929,93,3,3.23
2,0003,cli-0001,2026-05-16 00:00:00,581,7,3,2933,4052,37,2,5.41
3,0004,cli-0001,2026-05-17 00:00:00,582,5,4,5343,6534,118,9,7.63
4,0005,cli-0001,2026-05-18 00:00:00,581,2,3,1767,3167,16,1,6.25


In [7]:
# 06 profile_metrics 검증

print("행 수:", len(profile_metrics))
print("기간:", profile_metrics["profile_metric_at"].min(), "~", profile_metrics["profile_metric_at"].max())
print("고객사 수:", profile_metrics["client_id"].nunique())

# 논리 검증
print("노출 수 < 도달 수 오류:", (profile_metrics["profile_impressions_count"] < profile_metrics["profile_reach_count"]).sum())
print("웹사이트 클릭 수 > 프로필 방문 수 오류:", (profile_metrics["profile_website_click_count"] > profile_metrics["profile_visit_count"]).sum())
print("신규 팔로워 음수 오류:", (profile_metrics["profile_new_followers_count"] < 0).sum())

profile_metrics.describe()

행 수: 32
기간: 2026-05-14 00:00:00 ~ 2026-06-14 00:00:00
고객사 수: 1
노출 수 < 도달 수 오류: 0
웹사이트 클릭 수 > 프로필 방문 수 오류: 0
신규 팔로워 음수 오류: 0


,profile_followers_count,profile_new_followers_count,profile_unfollowers_count,profile_reach_count,profile_impressions_count,profile_visit_count,profile_website_click_count,profile_website_click_rate
count,32.000000,32.000000,32.00000,32.000000,32.000000,32.000000,32.000000,32.000000
mean,683.906250,12.093750,3.65625,6975.750000,11187.750000,158.937500,12.468750,7.422813
std,87.977768,8.931132,2.61027,4183.006551,7147.804513,120.799471,11.042394,2.769932
min,574.000000,1.000000,0.00000,1002.000000,1974.000000,16.000000,1.000000,2.330000
25%,602.000000,5.000000,2.00000,3651.250000,5247.250000,67.500000,4.750000,5.582500
50%,678.500000,8.500000,3.50000,6273.000000,9440.000000,125.000000,9.000000,7.025000
75%,745.750000,18.250000,5.25000,9807.000000,16867.250000,226.250000,16.250000,9.607500
max,841.000000,34.000000,9.00000,15935.000000,27388.000000,527.000000,41.000000,12.900000


In [8]:
# 07 CSV 저장

profile_metrics.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"저장 완료: {OUTPUT_PATH}")

저장 완료: ..\data\12_profile_metrics.csv
